# Loans and credit facilities

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Prerequisites:** `02_pricing/pricing_fundamentals.ipynb`. Native **`term_loan`** and **`revolving_credit`** JSON cover draws, commitment/usage fees, and delayed-draw style availability windows where modeled.


## Concept

- **Term loan:** fixed or floating margin; amortization rules.
- **Revolver:** deterministic draw/repay paths; **commitment** and **usage** fees on drawn vs undrawn.
- **DDTL:** delayed-draw tranche inside `term_loan` encodes availability windows.

**IRR** (money-weighted return) is the rate that zeros PV of all cashflows—conceptually parallel to **discount margin** / **all-in yield** metrics when exposed.


In [ ]:
import json
from datetime import date, timedelta

from finstack_quant.valuations import ValuationResult
from finstack_quant.valuations.instruments import price_instrument_with_metrics, validate_instrument_json
from finstack_quant.core.market_data import DiscountCurve, ForwardCurve, MarketContext, ScalarTimeSeries

print("Imports OK (loans / RCF).")


## Instrument JSON

**RCF recovery:** For a **senior secured** revolving facility, a **nonzero recovery rate** (here **70%**) is typical in credit-adjusted PV; `0.0` would overstate loss given default in a secured context.


In [ ]:
import json
from pathlib import Path

_NOTEBOOK_DATA = json.loads(Path("data/loans_and_credit_facilities.json").read_text())

AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

term_loan = _NOTEBOOK_DATA['term_loan']

revolver = _NOTEBOOK_DATA['revolver']

term_json = json.dumps(term_loan)
rev_json = json.dumps(revolver)
for label, js in (("term_loan", term_json), ("revolving_credit", rev_json)):
    validate_instrument_json(js)
    print(label, "JSON OK")

print(
    "IRR concept: solve r with sum(CF_t / (1+r)^t)=0 on the facility cashflow schedule (fees + interest + draws/repays)."
)


## MarketContext


In [ ]:
ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.5, 0.985), (1.0, 0.97), (3.0, 0.90), (5.0, 0.82), (10.0, 0.65)],
    day_count="act_365f",
)
sofr = ForwardCurve(
    "USD-SOFR-3M",
    0.25,
    knots=[(0.0, 0.052), (1.0, 0.048), (3.0, 0.045), (5.0, 0.043), (10.0, 0.041)],
    base_date=AS_OF,
    day_count="act_360",
)
fixing_start = date(2024, 1, 1)
sofr_fixings = [
    (fixing_start + timedelta(days=offset), 0.05)
    for offset in range((AS_OF - fixing_start).days + 1)
]
mc = MarketContext().insert(ois).insert(sofr)
mc.insert_series(ScalarTimeSeries("FIXING:USD-SOFR-3M", sofr_fixings))
market_json = mc.to_json()
print("market ready")


## Pricing


In [ ]:
for label, js in (("term_loan", term_json), ("revolving_credit", rev_json)):
    out = price_instrument_with_metrics(js, market_json, AS_OF_STR, model="discounting")
    print(label, ValuationResult.from_json(out))


## Metrics


In [ ]:
for label, js, metrics in (
    ("term_loan", term_json, ["dv01", "effective_rate"]),
    ("revolving_credit", rev_json, ["dv01", "discount_margin", "effective_rate"]),
):
    out = price_instrument_with_metrics(
        js, market_json, AS_OF_STR, model="discounting", metrics=metrics
    )
    vr = ValuationResult.from_json(out)
    print("—", label)
    for metric in metrics:
        value = vr.get_metric(metric)
        if value is not None:
            print(f"  {metric}: {value}")


## Takeaways

- **Term loans** and **RCFs** share the same JSON pricing pipeline as simpler rates instruments.
- **Fee stacks** (commitment / usage / facility) interact with draws to shape IRR and PV.
- If a schema field moves between releases, **`validate_instrument_json`** is the first line of defense.
